In [1]:
# Imports
import sys, time, math, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd().parent))

from common.utils import set_seed, get_device, count_parameters
from common.datasets import (FASHION_MNIST_CLASSES, FASHION_MNIST_SHAPE, 
                             load_fashion_mnist, load_or_create_split,
                             pixel_stats, to_tensor_dataset,
                             make_train_loader, make_eval_loaders)
from common.training import fit
from common.evaluation import collect_outputs, evaluate_model, format_results_table
from common.plotting import setup_style, save_fig, plot_learning_curves, plot_confusion_matrices
from E1.config import CONFIG, SPLIT_PATH, NUM_CLASSES

set_seed(CONFIG["seed"])
DEVICE = get_device()

E2_CONFIG = dict(
  embed_dim=64, 
  depth=2, 
  num_heads=4, 
  mlp_ratio=2, 
  dropout=0.1, 
  pool="cls", 
  head="softmax"
)

RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
setup_style()

In [2]:
# load data
train_images, train_labels, test_images, test_labels = load_fashion_mnist(CONFIG["data_dir"])
train_idx, val_idx = load_or_create_split(train_labels, SPLIT_PATH, CONFIG["seed"], CONFIG["val_ratio"])
X_train, y_train = train_images[train_idx], train_labels[train_idx]
X_val, y_val = train_images[val_idx], train_labels[val_idx]
X_test, y_test = test_images, test_labels
mean, std = pixel_stats(X_train)
datasets = {
    "train": to_tensor_dataset(X_train, y_train, mean, std),
    "val": to_tensor_dataset(X_val, y_val, mean, std),
    "test": to_tensor_dataset(X_test, y_test, mean, std),
}

train_loader = make_train_loader(datasets["train"], CONFIG["batch_size"], CONFIG["seed"])
eval_loaders = make_eval_loaders(datasets, CONFIG["batch_size"])

## MSA

Input $X \in \mathbb{R}^{B \times N \times D}$, gồm $N$ token, mỗi token có $D$ chiều, chia thành $h$ head với $d = D/h$, $B$ là số lượng sample.

Giả sử ta đem các giá trị vào một Tensor, với B là số lượng sample, N là số lượng token, thì:

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$
$$\text{head}_i = \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d}}\right) V_i$$
$$\text{MSA}(X) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)\, W_O$$

Cụ thể, $Q,K,V \in \mathbb{R}^{B \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times N \times N}$ hoặc $Q,K,V \in \mathbb{R}^{B \times h \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times h \times N \times N}$ 

$\text{head}_i \in \mathbb{R}^{B \times N \times d}$

In [3]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.embed_dim = embed_dim                # D
        self.num_heads = num_heads                # h
        self.head_dim = embed_dim // num_heads    # D/h
        self.scale = 1 / math.sqrt(self.head_dim)
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=bias) # W_q, W_k, W_v combined
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias) # Output projection
        self.attn_dropout = nn.Dropout(attn_dropout)

    def forward(self, x, return_attn: bool = False):
        B, N, D = x.shape

        # 1. Tính Q, K, V
        qkv = self.qkv(x)  # (B, N, 3*D)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # (3, B, h, N, D/h)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each is (B, h, N, D/h)

        # 2. Tính attention scores bằng einsum
        attn_scores = torch.einsum('bhqd,bhkd->bhqk', q, k) * self.scale  # (B, h, N, N)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)

        # 3. Tính weighted sum của V
        attn_output = torch.einsum('bhqk,bhkd->bhqd', attn_probs, v)  # (B, h, N, D/h)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(B, N, D)  # (B, N, D)

        # 4. Áp dụng output projection
        output = self.out_proj(attn_output)

        if return_attn:
            return output, attn_probs
        return output


In [4]:
class MSAPyTorch(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=attn_dropout, 
            bias=bias, 
            batch_first=True # Để đầu vào có dạng (B, N, D) thay vì (N, B, D)
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        attn_output, attn_weights = self.mha(x, x, x, need_weights=return_attn, average_attn_weights=False)
        if return_attn:
            return attn_output, attn_weights
        return attn_output

In [5]:
ATTENTIONS = {
  "scratch": MultiHeadSelfAttention,
  "torch": MSAPyTorch,
}

# Ánh xạ hậu tố key: scratch -> torch
_MSA_KEY_MAP = {
  "qkv.weight": "mha.in_proj_weight",
  "qkv.bias": "mha.in_proj_bias",
  "out_proj.weight": "mha.out_proj.weight",
  "out_proj.bias": "mha.out_proj.bias",
}

def scratch_to_torch_state_dict(state_dict: dict) -> dict:
    torch_state_dict = {}
    for key, value in state_dict.items():
        # Nếu key kết thúc bằng một hậu tố trong _MSA_KEY_MAP, ánh xạ sang key tương ứng của torch
        for suffix, torch_key in _MSA_KEY_MAP.items():
            if key.endswith(suffix):
                torch_state_dict[key[:-len(suffix)] + torch_key] = value
                break
        else:
            torch_state_dict[key] = value
    return torch_state_dict

In [6]:
# Kiểm tra hai mô hình Attention

@torch.no_grad()
def check_scratch_and_torch(B=4, N=20, D=64, h=4, atol=1e-5):
    set_seed(0)
    scratch = MultiHeadSelfAttention(embed_dim=D, num_heads=h).eval()
    torch_msa = MSAPyTorch(embed_dim=D, num_heads=h).eval()
    torch_msa.load_state_dict(scratch_to_torch_state_dict(scratch.state_dict()))

    # Tạo đầu vào ngẫu nhiên
    x = torch.randn(B, N, D)

    # Kiểm tra đầu ra của scratch và torch có gần nhau không
    out_scratch, attn_scratch = scratch(x, return_attn=True)
    out_torch, attn_torch = torch_msa(x, return_attn=True)
    assert torch.allclose(out_scratch, out_torch, atol=atol) and torch.allclose(attn_scratch, attn_torch, atol=atol), "Outputs are not close!"

    # Tính số tham số
    n_s, n_t = count_parameters(scratch), count_parameters(torch_msa)
    print(f"Number of parameters - Scratch: {n_s}, Torch: {n_t}")

    # Kiểm tra số tham số phải bằng nhau và bằng 4D^2 + 4D
    assert n_s == n_t == 4 * D**2 + 4 * D, "Number of parameters is not correct!"

check_scratch_and_torch()

Number of parameters - Scratch: 16640, Torch: 16640


In [7]:
# Đo tốc độ chạy của hai mô hình Attention
def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

def bench_attention(cls, N, D=64, h=4, B=128, iters=50, warmup=10, device=DEVICE):
    set_seed(0)
    module = cls(embed_dim=D, num_heads=h).to(device).train()
    x = torch.randn(B, N, D, device=device, requires_grad=True)
    for i in range(warmup + iters):
        if (i == warmup):
            sync(device)
            t0 = time.perf_counter()
        module(x).sum().backward()
    sync(device)
    t1 = time.perf_counter()
    return (t1 - t0) / iters * 1e3  # Convert to milliseconds

# N = số token với ảnh 28×28:
#   16  = patch size 7 → lưới 4×4
#   28  = mỗi hàng là 1 token
#   49  = patch size 4 → lưới 7×7
#   196 = patch size 2 → lưới 14×14
for N in (16, 28, 49, 196):
    times = {
        name: bench_attention(cls, N) for name, cls in ATTENTIONS.items()
    }
    print(f"N={N:>4}: " + "  ".join(f"{k}={v:6.2f} ms" for k, v in times.items()))

N=  16: scratch=  2.00 ms  torch=  1.92 ms
N=  28: scratch=  1.77 ms  torch=  2.51 ms
N=  49: scratch=  4.72 ms  torch=  5.01 ms
N= 196: scratch= 40.25 ms  torch= 39.27 ms


In [8]:
class FeedForward(nn.Module):
    """Áp dụng độc lập cho từng token: [B, N, D] -> [B, N, D]"""

    def __init__(self, embed_dim, mlp_ratio=2, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(p=dropout)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, dropout=0.1, attention="scratch"):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = ATTENTIONS[attention](embed_dim=embed_dim, num_heads=num_heads, attn_dropout=dropout)
        self.drop = nn.Dropout(p=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim=embed_dim, mlp_ratio=mlp_ratio, dropout=dropout)

    def forward(self, x, return_attn=False):
        # Pre-norm and attention
        if return_attn:
            attn_out, attn_weights = self.attention(self.norm1(x), return_attn=True)
        else:
            attn_out, attn_weights = self.attention(self.norm1(x)), None
        
        x = x + self.drop(attn_out)
        x = x + self.ffn(self.norm2(x))

        if return_attn:
            return x, attn_weights
        return x

In [10]:
from E1.classifiers import SoftmaxClassifier, MLPClassifier

class TransformerClassifier(nn.Module):
    def __init__(self, tokenizer, attention="scratch", embed_dim=64, depth=2, num_heads=4, 
                 mlp_ratio=2, dropout=0.1, pool="cls", head="softmax", use_pos=True):
        super().__init__()
        self.tokenizer = tokenizer
        self.pool = pool
        num_tokens = tokenizer.num_tokens
        if pool == "cls":
            num_tokens += 1

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if pool == "cls" else None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim)) if use_pos else None
        self.pos_drop = nn.Dropout(p=dropout) if use_pos else None
        self.blocks = nn.ModuleList([
            EncoderBlock(embed_dim=embed_dim, num_heads=num_heads, 
                         mlp_ratio=mlp_ratio, dropout=dropout, attention=attention)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        if head == "softmax":
            self.head = SoftmaxClassifier(input_dim=embed_dim, num_classes=NUM_CLASSES)
        else:
            self.head = MLPClassifier(input_dim=embed_dim, num_classes=NUM_CLASSES, 
                                      hidden_dims=(embed_dim * mlp_ratio,), dropout=dropout)
        
        if self.pos_embed is not None:
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.cls_token is not None:
            nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x, return_attn: bool = False):
        tokens = self.tokenizer(x)
        
        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(tokens.size(0), -1, -1)
            tokens = torch.cat((cls_tokens, tokens), dim=1)
        
        if self.pos_embed is not None:
            tokens = tokens + self.pos_embed
            tokens = self.pos_drop(tokens)

        attn_maps = []
        for block in self.blocks:
            if return_attn:
                tokens, attn = block(tokens, return_attn=True)
                attn_maps.append(attn)
            else:
                tokens = block(tokens)
        
        tokens = self.norm(tokens)
        
        if self.pool == "cls":
            tokens = tokens[:, 0]
        else:
            tokens = tokens.mean(dim=1)

        logits = self.head(tokens)
        if return_attn:
            return logits, attn_maps
        return logits

    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self(x), dim=-1)

In [11]:
# Tokenizer

class PatchEmbedding(nn.Module):
    """Tokenizer theo phương pháp Patching, default là patch_size=7, embed_dim=64"""
    def __init__(self, patch_size=7, embed_dim=64, img_size=28, in_channels=1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.img_size = img_size
        self.in_channels = in_channels
        self.num_tokens = (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size * in_channels, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size

        # [B, C, H/p, p, W/p, p] -> [B, H/p, W/p, C, p, p]
        x = x.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(B, -1, C * p * p)
        return self.proj(x)

class RowEmbedding(nn.Module):
    """Tokenizer theo phương pháp Row Embedding, với token size đúng bằng 28pixel - tức là một dòng của mỗi ảnh"""
    def __init__(self, embed_dim=64, img_size=28, in_channels=1):
        super().__init__()
        self.num_tokens = img_size # mỗi dòng là một token
        self.proj = nn.Linear(img_size * in_channels, embed_dim) # mỗi dòng của ảnh được flatten và ánh xạ vào không gian embedding
        # tương đương L(C·W → D)

    def forward(self, x):
        B, C, H, W = x.shape
        # Đưa H lên trước C: [B, C, H, W] -> [B, H, C, W]
        # [B, H, C, W] -> [B, H, C*W]
        # rồi chiếu sang không gian embedding D
        x = x.permute(0, 2, 1, 3)  # [B, C, H, W] -> [B, H, C, W]
        x = x.reshape(B, H, C * W)  # [B, H, C, W] -> [B, H, C*W]
        return self.proj(x)

from E1.classifiers import conv_block  

class CNNStemEmbedding(nn.Module):
    """Tokenizer theo phương pháp CNN Stem"""
    def __init__(self, embed_dim=64, img_size=28, input_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(input_channels, embed_dim // 2),
            conv_block(embed_dim // 2, embed_dim),
        )
        self.num_tokens = (img_size // 4) ** 2  # giả sử CNN stem giảm kích thước ảnh xuống 1/4
        self.proj = nn.Linear(embed_dim, embed_dim)  # chiếu các token CNN stem sang không gian embedding D


    def forward(self, x):
        f = self.features(x)  # [B, C', H', W']
        # flatten feature map: [B, C', H', W'] -> [B, H'*W', C']
        B, C, H, W = f.shape
        f = f.reshape(B, C, H * W)
        f = f.permute(0, 2, 1)  # [B, C, H*W] -> [B, H*W, C]
        return self.proj(f)

TOKENIZERS = {
    "patch7": lambda D: PatchEmbedding(patch_size=7, embed_dim=D),
    "row":    lambda D: RowEmbedding(embed_dim=D),
    "cnn":    lambda D: CNNStemEmbedding(embed_dim=D),
}

In [12]:
# Test 1 shape và bảng tham số

for name, make in TOKENIZERS.items():
    set_seed(0)
    model = TransformerClassifier(tokenizer=make(64))
    x = torch.randn(8, 1, 28, 28)

    print(f"Testing tokenizer: {name}")
    assert model(x).shape == (8, 10), "Output shape is incorrect"

    print(f"{'name':<10} {'parameters':>8}")
    for name, part in [("tokenizer", model.tokenizer), ("1 block", model.blocks[0]),
                      ("  ├ attn", model.blocks[0].attention), ("  └ ffn", model.blocks[0].ffn),
                      ("head", model.head), ("TOTAL", model)]:
        print(f"{name:<10} {count_parameters(part):>8,}")

    print()

Testing tokenizer: patch7
name       parameters
tokenizer     3,200
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        72,074

Testing tokenizer: row
name       parameters
tokenizer     1,856
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        71,498

Testing tokenizer: cnn
name       parameters
tokenizer    22,976
1 block      33,472
  ├ attn     16,640
  └ ffn      16,576
head            650
TOTAL        93,962



In [13]:
# Test 2 — scratch và torch cho cùng logits khi dùng chung trọng số

@torch.no_grad()
def check_classifier_equivalence(atol=1e-5, tokenizer_name="patch7"):
    set_seed(0)
    tok = TOKENIZERS[tokenizer_name](64)
    m_scratch = TransformerClassifier(tok, attention="scratch").eval()
    m_torch = TransformerClassifier(tok, attention="torch").eval()
    m_torch.load_state_dict(scratch_to_torch_state_dict(m_scratch.state_dict()))   # strict=True
    x = torch.randn(8, 1, 28, 28)
    diff = (m_scratch(x) - m_torch(x)).abs().max().item()
    print(f"{tokenizer_name:<8} max |Δlogits| = {diff:.2e}")
    assert diff < atol, f"Difference {diff:.2e} exceeds tolerance {atol}"

In [14]:
for tokenizer_name in TOKENIZERS.keys():
    check_classifier_equivalence(tokenizer_name=tokenizer_name)

patch7   max |Δlogits| = 7.15e-07
row      max |Δlogits| = 6.41e-07
cnn      max |Δlogits| = 4.17e-07


In [15]:
# Test 3 — overfit một batch

for name, make in TOKENIZERS.items():
    print(f"Testing tokenizer: {name}")
    xb, yb = next(iter(train_loader))
    xb, yb = xb[:64], yb[:64]
    D = 64
    tok = make(D)
    set_seed(0)
    model = TransformerClassifier(tok)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    for step in range(200):
      model.train()
      opt.zero_grad()
      logits = model(xb)
      loss = F.cross_entropy(logits, yb)
      loss.backward()
      opt.step()
      if step % 50 == 0 or step == 199:
        print(f"Step {step:>3}: loss = {loss.item():.4f}")
    assert loss.item() < 0.05, f"Không overfit được 1 batch: loss = {loss.item():.4f}"
    print()

Testing tokenizer: patch7
Step   0: loss = 2.5879
Step  50: loss = 0.2056
Step 100: loss = 0.0484
Step 150: loss = 0.0276
Step 199: loss = 0.0094

Testing tokenizer: row
Step   0: loss = 2.3514
Step  50: loss = 0.1912
Step 100: loss = 0.0281
Step 150: loss = 0.0261
Step 199: loss = 0.0071

Testing tokenizer: cnn
Step   0: loss = 2.5510
Step  50: loss = 0.4447
Step 100: loss = 0.0647
Step 150: loss = 0.0242
Step 199: loss = 0.0081



## Training

In [16]:
import copy

@torch.no_grad()
def measure_latency(model, iters=200, warmup=10, device=torch.device("cpu")):
    """Đo độ trễ của một model với batch=1. Chọn device mặc định là cpu để gần với kịch bản chạy trên thiết bị."""
    model = copy.deepcopy(model).to(device).eval()
    x = torch.randn(1, 1, 28, 28, device=device)

    # warm up
    for _ in range(warmup):
        _ = model(x)

    sync(device)
    t0 = time.perf_counter()
    
    for _ in range(iters):
        _ = model(x)
    
    sync(device)
    t1 = time.perf_counter()

    return 1000 * (t1 - t0) / iters  # latency in milliseconds

In [17]:
def run_epoch(tokenizer_name, attention="scratch", init_state=None, seed=CONFIG['seed']):
    assert attention in ["scratch", "torch"], "Attention must be either 'scratch' or 'torch'"
    name = f"{tokenizer_name}_{attention}"
    set_seed(seed)
    D = E2_CONFIG["embed_dim"]
    model = TransformerClassifier(
        tokenizer=TOKENIZERS[tokenizer_name](D),
        attention=attention,
        embed_dim=D,
        depth=E2_CONFIG["depth"],
        num_heads=E2_CONFIG["num_heads"],
        mlp_ratio=E2_CONFIG["mlp_ratio"],
        dropout=E2_CONFIG["dropout"],
        pool=E2_CONFIG["pool"],
        head=E2_CONFIG["head"]
    )

    if init_state is not None:
        model.load_state_dict(init_state)

    init_snapshot = {
        k: v.detach().clone() for k, v in model.state_dict().items()
    }

    train_loader = make_train_loader(datasets["train"], CONFIG["batch_size"], CONFIG["seed"])
    t0 = time.perf_counter()
    history, best_epoch = fit(
        model=model,
        train_loader=train_loader,
        val_loader=eval_loaders["val"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        seed=seed,
        patience=CONFIG["patience"],
        device=DEVICE,
    )
    history["train_time"] = time.perf_counter() - t0

    row = evaluate_model(
        model=model,
        name=name,
        best_epoch=best_epoch,
        train_time=history["train_time"],
        eval_loaders=eval_loaders,
        device=DEVICE,
    )
    row.update(
        tokenizer=tokenizer_name,
        attention=attention,
        num_tokens=model.tokenizer.num_tokens,
        sec_per_epoch=float(np.mean(history["epoch_time"])),
        latency_ms=measure_latency(model),
    )

    outputs = collect_outputs(
        model=model,
        dataloader=eval_loaders["test"],
        device=DEVICE,
    )

    return row, history, outputs, model, init_snapshot

In [18]:
results, histories, outputs, models = [], {}, {}, {}
results_path, hist_path = RESULTS_DIR / "results.json", RESULTS_DIR / "histories.json"
results = json.loads(results_path.read_text()) if results_path.exists() else []
histories = json.loads(hist_path.read_text()) if hist_path.exists() else {}
done = {r["model"] for r in results}

def save_progress():
    results_path.write_text(json.dumps(results, indent=2))
    hist_path.write_text(json.dumps(histories, indent=2))

def record(name, row, hist, out, model):
    results.append(row)
    done.add(name)
    histories[name] = hist
    outputs[name] = out
    models[name] = model
    save_progress()
    torch.save(model.state_dict(), RESULTS_DIR / f"model_{name}.pt")
    np.savez(RESULTS_DIR / f"outputs_{name}.npz", y_true=out[0], y_pred=out[1], probs=out[2])

In [19]:
for tokenizer_name in TOKENIZERS:
    scratch_name, torch_name = f"{tokenizer_name}_scratch", f"{tokenizer_name}_torch"
    init_path = RESULTS_DIR / f"init_state_{scratch_name}.pt"

    # (1) Chạy toàn bộ tokenizer với cùng attention 'scratch' trước
    if scratch_name in done:
        print("Đã chạy ", scratch_name)
    else:
        row, hist, out, model, init_st = run_epoch(tokenizer_name, attention="scratch")
        torch.save(init_st, init_path)
        record(scratch_name, row, hist, out, model)

    # (2) Chạy toàn bộ với attention 'torch'
    if torch_name in done:
        print("Đã chạy ", torch_name)
    else:
        init_st = torch.load(init_path)
        row, hist, out, model, _ = run_epoch(tokenizer_name, attention="torch", init_state=scratch_to_torch_state_dict(init_st))
        record(torch_name, row, hist, out, model)
    
    # Dùng format_results_table ở E1
    print(format_results_table(results))

Epoch 1/50 - Train loss: 0.8320, Train acc: 0.7056 - Val loss: 0.5506, Val acc: 0.8016
Epoch 2/50 - Train loss: 0.5462, Train acc: 0.8039 - Val loss: 0.4604, Val acc: 0.8378
Epoch 3/50 - Train loss: 0.4819, Train acc: 0.8247 - Val loss: 0.4371, Val acc: 0.8431
Epoch 4/50 - Train loss: 0.4488, Train acc: 0.8364 - Val loss: 0.3991, Val acc: 0.8571
Epoch 5/50 - Train loss: 0.4235, Train acc: 0.8444 - Val loss: 0.3898, Val acc: 0.8595
Epoch 6/50 - Train loss: 0.4064, Train acc: 0.8518 - Val loss: 0.3912, Val acc: 0.8587
Epoch 7/50 - Train loss: 0.3947, Train acc: 0.8549 - Val loss: 0.3515, Val acc: 0.8716
Epoch 8/50 - Train loss: 0.3821, Train acc: 0.8593 - Val loss: 0.3630, Val acc: 0.8667
Epoch 9/50 - Train loss: 0.3737, Train acc: 0.8636 - Val loss: 0.3632, Val acc: 0.8693
Epoch 10/50 - Train loss: 0.3610, Train acc: 0.8682 - Val loss: 0.3390, Val acc: 0.8788
Epoch 11/50 - Train loss: 0.3548, Train acc: 0.8682 - Val loss: 0.3407, Val acc: 0.8783
Epoch 12/50 - Train loss: 0.3486, Train a